# 22. Memorization Analysis（spec §20）

## 学習目標
- 「学習データの再現」と「汎化」を区別する
- train文書の接頭辞からの greedy 継続が、val文書（対照）よりどれだけ原文に一致するかを測る
- 低い val loss だけでは汎化を証明できないことを理解する

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
M2_RUN = ROOT / "experiments/runs/m2_model_m_classical_seed42"
L_RUN = ROOT / "experiments/runs/m4_model_l_modern_seed42"
M5 = ROOT / "experiments/analysis_m5"

In [2]:
mem = load_json(M5 / "memorization.json")
tr, va = mem["train"], mem["validation"]
print(f"prefix {mem['prefix_len']} tokens → greedy continue {mem['continue_len']} tokens")
df = pd.DataFrame({
    "split": ["train (学習済み)","validation (未学習・対照)"],
    "mean_exact_match": [tr.get("mean_exact_match"), va.get("mean_exact_match")],
    "mean_lcs": [tr.get("mean_lcs"), va.get("mean_lcs")],
    "max_exact_match": [tr.get("max_exact_match"), va.get("max_exact_match")],
    "n": [tr.get("n"), va.get("n")],
})
df.round(2)

prefix 40 tokens → greedy continue 60 tokens


,split,mean_exact_match,mean_lcs,max_exact_match,n
0,train (学習済み),0.65,2.20,8,40
1,validation (未学習・対照),0.57,1.92,3,40


In [3]:
fig = go.Figure()
fig.add_trace(go.Bar(x=["train","validation"],
                     y=[tr.get("mean_exact_match"), va.get("mean_exact_match")],
                     marker_color=["#d62728","#1f77b4"], name="mean exact-match"))
fig.update_layout(title="接頭辞からの greedy 継続が原文と一致した token 数（train vs val）",
                  yaxis_title="mean exact-match length", template="plotly_white", height=360)
fig.show()
gap = (tr.get("mean_exact_match") or 0) - (va.get("mean_exact_match") or 0)
print(f"train − val exact-match gap = {gap:.2f} tokens")
print(mem["note"])

train − val exact-match gap = 0.08 tokens
train >> validation on exact-match ⇒ memorization; similar ⇒ generalization


## Observation / Interpretation / Caveat
- **Observation**: train文書と val文書の exact-match 継続長は上のセルが実測。両者が近ければ暗記は弱く、train が大きく上回れば暗記のサイン。
- **Interpretation**: Model L は main を約0.9エポックしか見ておらず、逐語暗記は限定的なはず（train≈val なら汎化優位）。M1（45エポック）と対比すると、エポック数が暗記に効くことが分かる。
- **Caveat**: exact-match は暗記の一側面（近似再現は捉えない）。文書サンプルは各コーパスの先頭付近に限定。greedy 継続なので、サンプリングでは一致率が下がる。「低 val loss=汎化」は依然として証明にならない。

**次へ**: [23_sft_analysis](23_sft_analysis.ipynb)